<a href="https://colab.research.google.com/github/orbitalaerospace/CUDA-Learning/blob/main/CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun Sep 27 19:20:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
PyTorch CUDA: 12.8
GPU: Tesla T4


In [ ]:
$%%writefile hello.cu
#include <stdio.h>
__global__ void helloFromGPU(){
  printf("Hello from GPU! Thread %d\n", threadIdx.x);
}
int main(){
  helloFromGPU<<<1,4>>>();
  cudaDeviceSynchronize();
  return 0;
}


Writing hello.cu


In [ ]:
!nvcc hello.cu -o hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./hello

Hello from GPU! Thread 0
Hello from GPU! Thread 1
Hello from GPU! Thread 2
Hello from GPU! Thread 3


In [ ]:
%%writefile blocks.cu
#include <stdio.h>
__global__ void hellofromgpu(){
    printf("Block: %d, Thread: %d\n",blockIdx.x, threadIdx.x);
}
int main(){
    hellofromgpu<<<2,4>>>();
    cudaDeviceSynchronize();
    return 0;
}

Writing blocks.cu


In [ ]:
!nvcc blocks.cu -o blocks

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./blocks

Block: 1, Thread: 0
Block: 1, Thread: 1
Block: 1, Thread: 2
Block: 1, Thread: 3
Block: 0, Thread: 0
Block: 0, Thread: 1
Block: 0, Thread: 2
Block: 0, Thread: 3


In [ ]:
%%writefile blockdim.cu
#include <stdio.h>
__global__ void blockdim(){
  printf("Number of Threads in Block : %d, is : %d\n", blockIdx.x, blockDim.x);
}
int main(){
  blockdim<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Overwriting blockdim.cu


In [ ]:
!nvcc blockdim.cu -o blockdim

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./blockdim

Number of Threads in Block : 1, is : 4
Number of Threads in Block : 1, is : 4
Number of Threads in Block : 1, is : 4
Number of Threads in Block : 1, is : 4
Number of Threads in Block : 0, is : 4
Number of Threads in Block : 0, is : 4
Number of Threads in Block : 0, is : 4
Number of Threads in Block : 0, is : 4


In [ ]:
%%writefile indexing.cu
#include <stdio.h>
__global__ void index(){
  printf("At the block : %d, while thread : %d is executing, the number of threads in the block is: %d\n", blockIdx.x, threadIdx.x, blockDim.x);
}
int main(){
  index<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Overwriting indexing.cu


In [ ]:
!nvcc indexing.cu -o indexing

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./indexing

At the block : 1, while thread : 0 is executing, the number of threads in the block is: 4
At the block : 1, while thread : 1 is executing, the number of threads in the block is: 4
At the block : 1, while thread : 2 is executing, the number of threads in the block is: 4
At the block : 1, while thread : 3 is executing, the number of threads in the block is: 4
At the block : 0, while thread : 0 is executing, the number of threads in the block is: 4
At the block : 0, while thread : 1 is executing, the number of threads in the block is: 4
At the block : 0, while thread : 2 is executing, the number of threads in the block is: 4
At the block : 0, while thread : 3 is executing, the number of threads in the block is: 4


In [ ]:
%%writefile globalid.cu
#include <stdio.h>
__global__ void global(){
    int globalid = blockIdx.x * blockDim.x + threadIdx.x;
    printf("The global id of Thread %d at block %d is %d when the number of threads is %d\n",threadIdx.x, blockIdx.x, globalid, blockDim.x);
}

int main(){
    global<<<2,4>>>();
    cudaDeviceSynchronize();
    return 0;
}

Overwriting globalid.cu


In [ ]:
!nvcc globalid.cu -o globalid

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./globalid

The global id of Thread 0 at block 0 is 0 when the number of threads is 4
The global id of Thread 1 at block 0 is 1 when the number of threads is 4
The global id of Thread 2 at block 0 is 2 when the number of threads is 4
The global id of Thread 3 at block 0 is 3 when the number of threads is 4
The global id of Thread 0 at block 1 is 4 when the number of threads is 4
The global id of Thread 1 at block 1 is 5 when the number of threads is 4
The global id of Thread 2 at block 1 is 6 when the number of threads is 4
The global id of Thread 3 at block 1 is 7 when the number of threads is 4


In [ ]:
%%writefile cudamalloc.cu
#include <stdio.h>
#include <cuda_runtime.h>
int main(){
  int x = 10;
  int *d_x;
  cudaMalloc(&d_x, sizeof(int));
  printf("CPU Value:%d", x);
  cudaFree(d_x);
  return 0;
}

Overwriting cudamalloc.cu


In [ ]:
!nvcc cudamalloc.cu -o cudamalloc

In [ ]:
!./cudamalloc

CPU Value:10

In [ ]:
%%writefile cudamemcpy.cu
#include <stdio.h>
#include <cuda_runtime.h>
int main(){
  int x = 10;
  int *d_x;
  int result;
  cudaMalloc(&d_x, sizeof(int));
  cudaMemcpy(d_x, &x, sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(&result, d_x, sizeof(int), cudaMemcpyDeviceToHost);
  printf("GPU Variable: %d", result);
  cudaFree(d_x);
  return 0;
}

Overwriting cudamemcpy.cu


In [ ]:
!nvcc cudamemcpy.cu -o cudamemcpy

In [ ]:
!./cudamemcpy

GPU Variable: 10

In [ ]:
%%writefile gpucompute.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void multiply(int *value){
  *value = *value * 2;
}

int main(){
  int x = 10;
  int *d_x;
  int result;
  cudaMalloc(&d_x, sizeof(int));
  cudaMemcpy(d_x, &x, sizeof(int), cudaMemcpyHostToDevice);
  multiply<<<1,1>>>(d_x);
  cudaDeviceSynchronize();
  cudaMemcpy(&result, d_x, sizeof(int), cudaMemcpyDeviceToHost);
  printf("The product is %d\n", result);
  cudaFree(d_x);
  return 0;
}

Overwriting gpucompute.cu


In [ ]:
!nvcc gpucompute.cu -o gpucompute

In [ ]:
!./gpucompute

The product is 20


In [ ]:
%%writefile vectormultiply.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectormul(int *values){
  values[threadIdx.x] = values[threadIdx.x] * 2;
}

int main(){
  int values[8] = {10, 20, 30, 40, 50, 60, 70, 80};
  int *d_x;
  cudaMalloc(&d_x, sizeof(values));
  cudaMemcpy(d_x, values, sizeof(values), cudaMemcpyHostToDevice);
  vectormul<<<1,8>>>(d_x);
  cudaDeviceSynchronize();
  cudaMemcpy(values, d_x, sizeof(values),cudaMemcpyDeviceToHost);
  for(int i = 0; i<8; i++){
    printf("%d\n", values[i]);
  }
  cudaFree(d_x);
  return 0;
}

Overwriting vectormultiply.cu


In [ ]:
!nvcc vectormultiply.cu -o vectormultiply

In [ ]:
!./vectormultiply

20
40
60
80
100
120
140
160


In [ ]:
%%writefile cuda_recap.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void mul(int *arr){
  arr[threadIdx.x] = arr[threadIdx.x] * 3;
}

int main(){
  int arr[8] = {5, 10, 15, 20, 25, 30, 35, 40};
  int *d_x;
  cudaMalloc(&d_x, sizeof(arr));
  cudaMemcpy(d_x, arr, sizeof(arr), cudaMemcpyHostToDevice);
  mul<<<1,8>>>(d_x);
  cudaDeviceSynchronize();
  cudaMemcpy(arr, d_x, sizeof(arr), cudaMemcpyDeviceToHost);
  for(int i = 0; i<8; i++){
    printf("Product of %d number is : %d\n", i, arr[i]);
  }
  cudaFree(d_x);
  return 0;
}


Overwriting cuda_recap.cu


In [ ]:
!nvcc cuda_recap.cu -o cuda_recap

In [ ]:
!./cuda_recap

Product of 0 number is : 15
Product of 1 number is : 30
Product of 2 number is : 45
Product of 3 number is : 60
Product of 4 number is : 75
Product of 5 number is : 90
Product of 6 number is : 105
Product of 7 number is : 120


In [ ]:
%%writefile bounds.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void mult(int *arr1, int n){
  int index = blockIdx.x * blockDim.x + threadIdx.x;

    arr1[index] = arr1[index] * 3;
}

int main(){
  int arr1[10] = {5, 10, 15, 20, 25, 30, 35, 40, 45, 50};
  int *d_x;
  cudaMalloc(&d_x, sizeof(arr1));
  cudaMemcpy(d_x, arr1, sizeof(arr1), cudaMemcpyHostToDevice);
  mult<<<2,8>>>(d_x, 10);
  cudaDeviceSynchronize();
  cudaMemcpy(arr1, d_x, sizeof(arr1), cudaMemcpyDeviceToHost);
  for(int i = 0; i<10; i++){
    printf("Product of %d number is : %d\n", i, arr1[i]);
  }
  cudaFree(d_x);
  return 0;
}


Overwriting bounds.cu


In [ ]:
!nvcc bounds.cu -o bounds

In [ ]:
!./bounds

Product of 0 number is : 15
Product of 1 number is : 30
Product of 2 number is : 45
Product of 3 number is : 60
Product of 4 number is : 75
Product of 5 number is : 90
Product of 6 number is : 105
Product of 7 number is : 120
Product of 8 number is : 45
Product of 9 number is : 50


In [4]:
%%writefile twod.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void twodim(){
  printf("Thread x axis: %d\n Thread y axis: %d\n", threadIdx.x, threadIdx.y);
}

int main(){
  twodim<<<1, dim3(3,2)>>>();
  cudaDeviceSynchronize();
  return 0;
  }

Overwriting twod.cu


In [5]:
!nvcc twod.cu -o twod

In [6]:
!./twod

Thread x axis: 0
 Thread y axis: 0
Thread x axis: 1
 Thread y axis: 0
Thread x axis: 2
 Thread y axis: 0
Thread x axis: 0
 Thread y axis: 1
Thread x axis: 1
 Thread y axis: 1
Thread x axis: 2
 Thread y axis: 1


In [1]:
%%writefile global2d.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void g2d(){
  int globalx = blockDim.x * blockIdx.x + threadIdx.x;
  int globaly = blockDim.y * blockIdx.y + threadIdx.y;
  printf("Global id of x: %d, Global id of y: %d\n", globalx, globaly);
}

int main(){
  g2d<<<dim3(2,2), dim3(2,3)>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing global2d.cu


In [2]:
!nvcc global2d.cu -o global2d

In [3]:
!./global2d

Global id of x: 0, Global id of y: 3
Global id of x: 1, Global id of y: 3
Global id of x: 0, Global id of y: 4
Global id of x: 1, Global id of y: 4
Global id of x: 0, Global id of y: 5
Global id of x: 1, Global id of y: 5
Global id of x: 0, Global id of y: 0
Global id of x: 1, Global id of y: 0
Global id of x: 0, Global id of y: 1
Global id of x: 1, Global id of y: 1
Global id of x: 0, Global id of y: 2
Global id of x: 1, Global id of y: 2
Global id of x: 2, Global id of y: 0
Global id of x: 3, Global id of y: 0
Global id of x: 2, Global id of y: 1
Global id of x: 3, Global id of y: 1
Global id of x: 2, Global id of y: 2
Global id of x: 3, Global id of y: 2
Global id of x: 2, Global id of y: 3
Global id of x: 3, Global id of y: 3
Global id of x: 2, Global id of y: 4
Global id of x: 3, Global id of y: 4
Global id of x: 2, Global id of y: 5
Global id of x: 3, Global id of y: 5
